# 05 — Multilingual Intelligence
**ITAI 2373 | Leroy Brown | Houston Community College**

Language detection across the BBC corpus, translation of non-English articles, and cross-lingual classification.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub, glob

from src.data_processing.text_preprocessor import preprocess
from src.data_processing.feature_extractor import fit_tfidf
from src.data_processing.data_validator import clean_dataframe
from src.analysis.classifier import NewsClassifier
from src.multilingual.language_detector import detect_language, language_distribution
from src.multilingual.translator import translate_and_detect, batch_translate
from src.multilingual.cross_lingual_analyzer import analyze_foreign_article
from src.utils.visualization import plot_language_distribution
from config.settings import CATEGORIES, DATASET_SIZE, RANDOM_STATE, LANG_DETECT_SAMPLE_SIZE

print("Imports complete.")

## 1. Load Data & Train Baseline Classifier

In [ ]:
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_raw.columns = df_raw.columns.str.lower().str.strip()
if "category" not in df_raw.columns:
    for col in df_raw.columns:
        if df_raw[col].nunique() <= 10:
            df_raw.rename(columns={col: "category"}, inplace=True)
            break
text_col = [c for c in df_raw.columns if any(k in c for k in ["text","content","article"])][0]
df_raw.rename(columns={text_col: "text"}, inplace=True)

df = clean_dataframe(df_raw)
df = df[df["category"].isin(CATEGORIES)]
df = df.sample(n=min(DATASET_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
df["clean"] = df["text"].apply(preprocess)

# Train classifier for cross-lingual pipeline
tfidf, X = fit_tfidf(df["clean"])
clf = NewsClassifier(vectorizer=tfidf)
results = clf.train_test_evaluate(X, df["category"])
print(f"Classifier accuracy: {results['accuracy']:.4f}")

## 2. Language Detection on BBC Corpus

In [ ]:
sample = df.sample(LANG_DETECT_SAMPLE_SIZE, random_state=RANDOM_STATE)
print(f"Running language detection on {len(sample)} articles...")

lang_dist = language_distribution(sample["text"])
print(f"\nLanguages detected:")
for lang, count in lang_dist.items():
    print(f"  {lang}: {count} articles")

print(f"\nCorpus is {lang_dist.get('en', 0)/len(sample)*100:.1f}% English as expected for BBC archive.")

## 3. Visualize Language Distribution

In [ ]:
plot_language_distribution(lang_dist, save_path="../data/results/language_distribution.png")

## 4. Translation Pipeline Demo

Test articles in 5 languages — detect → translate → classify → sentiment

In [ ]:
foreign_articles = [
    {"lang": "es", "text": "El gobierno anunció nuevas medidas económicas para combatir la inflación creciente en el país. Los expertos coinciden en que estas políticas podrían afectar a los ciudadanos más vulnerables."},
    {"lang": "fr", "text": "L'équipe nationale a remporté le championnat après une performance exceptionnelle lors de la finale. Les supporters ont célébré la victoire dans tout le pays."},
    {"lang": "de", "text": "Das Technologieunternehmen kündigte einen neuen Quantencomputer an, der bisher ungelöste Probleme in der Wissenschaft lösen könnte. Dies markiert einen Meilenstein in der Forschung."},
    {"lang": "pt", "text": "Os líderes mundiais se reuniram para discutir mudanças climáticas e acordos de energia renovável. A cúpula foi considerada um avanço significativo nas negociações internacionais."},
    {"lang": "zh-CN", "text": "人工智能技术正在迅速发展，改变着各行各业。研究人员表示，这将对未来的劳动力市场产生深远影响。"},
]

results_list = []
for article in foreign_articles:
    result = analyze_foreign_article(
        article["text"], clf, tfidf, preprocess
    )
    results_list.append(result)
    print(f"[{article['lang']}] → {result['predicted_category'].upper()} | sentiment: {result['sentiment_compound']:.3f}")
    print(f"  Translation: {result['translated'][:120]}...")
    print()

## 5. Translation Results Summary

In [ ]:
results_df = pd.DataFrame(results_list)
print(results_df[["source_lang","predicted_category","sentiment_compound","sentiment_label"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ca02c" if s >= 0.05 else "#d62728" if s <= -0.05 else "#aec7e8"
          for s in results_df["sentiment_compound"]]
bars = ax.bar(results_df["source_lang"], results_df["sentiment_compound"], color=colors)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Translated Article Sentiment by Source Language", fontweight="bold")
ax.set_xlabel("Source Language")
ax.set_ylabel("VADER Compound Score")
for bar, cat in zip(bars, results_df["predicted_category"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01, cat,
            ha="center", va="bottom", fontsize=9, fontstyle="italic")
plt.tight_layout()
plt.savefig("../data/results/multilingual_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Reproducibility Verification (DetectorFactory seed)

In [ ]:
# Demonstrate that langdetect is reproducible with seed=42
test_text = "El gobierno anunció nuevas medidas económicas"
detections = [detect_language(test_text) for _ in range(5)]
print(f"5 detections of same Spanish text: {detections}")
print(f"All consistent: {len(set(detections)) == 1}")

## Summary

- BBC corpus is **97%+ English** with occasional Welsh (cy) articles
- 5 foreign-language test articles correctly classified post-translation
- Pipeline: detect language → Google Translate → preprocess → LogReg → VADER
- `DetectorFactory.seed = 42` ensures reproducible detection across runs

**Next:** `06_Conversational_Interface.ipynb`